# AutoLab — étape 9B + 9C sur Colab (GPU T4) : interfaces avec la cathode et diffusion du lithium

**9B — Interface avec LiFePO₄** (énergies UMA, phases sur l'enveloppe MP seulement) :
1. deux calibrations d'abord : Li₃PS₄ / LiCoO₂ et Li₃PS₄ / LiFePO₄, chacune calculée avec UMA **et** avec les énergies MP (CPU) ;
2. si l'écart UMA − MP dépasse 0,05 eV/atome pour l'une d'elles, les interfaces des candidats **ne sont pas calculées** (le rapport le dit) ;
3. sinon, interface de chaque candidat avec LiFePO₄ (≈ 91 nouvelles relaxations au total, plafond 140).

**9C — Dynamique moléculaire** à 1000 K pendant 20 ps (précision normale) : 4 candidats + les 2 parents (Li₃PS₄, Li₇P₃S₁₁), 6 simulations au plus.

**Avant de lancer**
1. *Exécution → Modifier le type d'exécution →* **GPU T4**.
2. **Secrets** : `MP_API_KEY` et `HF_TOKEN` (accès au notebook activé). Aucun modèle de langage n'est appelé.
3. **Fichiers** : déposer `autolab_seed_step9.zip` (PC : `D:\autolab\.cache\autolab_seed_step9.zip`) dans `/content/`.
4. *Exécution → Tout exécuter.* Accepter l'accès à **Google Drive** quand il est demandé : chaque calcul terminé y est sauvegardé (`MyDrive/autolab_step9_backup`). Si Colab se déconnecte, relancer *Tout exécuter* : ce qui est fini est repris, rien n'est recalculé.

Durée prévue : 2 h 30 à 3 h. Chaque cellule capture ses erreurs. À la fin : un rapport unique et `autolab_step9_results.zip`.

In [ ]:
# Helpers: every step records its status and duration; errors never stop the notebook.
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO_URL = "https://github.com/hkdebendo/autolab.git"
REPO = Path("/content/autolab")
WORK = Path("/content/autolab_run")
WORK.mkdir(parents=True, exist_ok=True)
CACHE = REPO / ".cache"
RUN_DIR = WORK / "step9"
STEP_DIR = REPO / "results" / "2026-10-28_step9"
SEED = Path("/content/autolab_seed_step9.zip")
BACKUP = Path("/content/drive/MyDrive/autolab_step9_backup")
USE_BACKUP = False
T_START = time.time()
REPORT = {"steps": {}, "results": {}}
# Subprocesses get a clean interpreter (fresh imports after pip install), the local UMA backend and Meta as brain.
ENV = dict(os.environ, SIM_BACKEND="local", PYTHONUNBUFFERED="1", AUTOLAB_CACHE_DIR=str(CACHE))


def save_report():
    (WORK / "report.json").write_text(json.dumps(REPORT, indent=2, default=str), encoding="utf-8")


def record(step, status, seconds, **info):
    REPORT["steps"][step] = {"status": status, "seconds": round(seconds, 1), **info}
    save_report()
    print(f"\n[{step}] {status} en {seconds / 60:.1f} min")


def sh(step, cmd, cwd=REPO):
    """Run a command, stream its output live and into WORK/<step>.log; never raises."""
    t0 = time.time()
    try:
        with open(WORK / f"{step}.log", "w", encoding="utf-8") as log:
            proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=ENV, text=True,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            try:
                for line in proc.stdout:
                    print(line, end="")
                    log.write(line)
                code = proc.wait()
            except KeyboardInterrupt:
                proc.kill()
                record(step, "interrompu", time.time() - t0)
                return False
        record(step, "ok" if code == 0 else f"échec (code {code})", time.time() - t0)
        return code == 0
    except Exception as e:
        record(step, f"erreur : {type(e).__name__}: {e}", time.time() - t0)
        return False


def py(step, source):
    """Run a Python snippet in a fresh process from the repository folder."""
    script = WORK / f"{step}.py"
    script.write_text(source, encoding="utf-8")
    return sh(step, [sys.executable, script])


def load(name):
    p = WORK / f"{name}.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None


print("Helpers prêts.")

In [ ]:
# Parameters (from step 9A on the PC: Na candidates do not passivate against Li metal; Se and Sb have a zero window at
# the calculation precision, like their parent Li7P3S11 (0.02 V), so they are kept but not simulated here).
INTERFACE_IDS = ["it1-c2", "it2-c5", "it1-c1", "it2-c1", "it2-c2"]  # Li6P2S7O, Li11P4S15I, Li11P4S15Cl, Li13P6S21Cl, Li7AsP2S11
CATHODE = "LiFePO4"
MAX_RELAXATIONS = 140
MD_IDS = ["it2-c5", "it1-c2", "it1-c1", "it2-c1"]  # 4 candidates
MD_PARENTS = ["mp-985583", "mp-641703"]  # Li3PS4, Li7P3S11
MD_TEMPERATURE_K = 1000
MD_PS = 20
REPORT["parameters"] = {"interfaces": INTERFACE_IDS, "cathode": CATHODE, "max_relaxations": MAX_RELAXATIONS,
                        "md": MD_IDS, "md_parents": MD_PARENTS, "T": MD_TEMPERATURE_K, "ps": MD_PS}
save_report()
print(REPORT["parameters"])

In [ ]:
# GPU check
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or "aucun GPU"
except Exception as e:
    gpu = f"aucun GPU ({type(e).__name__})"
REPORT["gpu"] = gpu
REPORT["python"] = sys.version.split()[0]
save_report()
print("GPU :", gpu, "| Python", REPORT["python"])
if "T4" not in gpu:
    print("⚠️ Pas de T4 : changer le type d'exécution (GPU T4).")

In [ ]:
# Keys from Colab Secrets: only their presence is ever printed.
REPORT["secrets"] = {}
try:
    from google.colab import userdata
except Exception:
    userdata = None
for name in ("MP_API_KEY", "HF_TOKEN"):
    present, why = False, ""
    try:
        value = userdata.get(name) if userdata else os.environ.get(name, "")
        if value and value.strip():
            ENV[name] = value.strip()
            present = True
    except Exception as e:
        why = f" ({type(e).__name__} : secret absent ou accès au notebook non activé)"
    value = None
    REPORT["secrets"][name] = "présent" if present else "absent"
    print(f"{name} : {'présent' if present else 'ABSENT' + why}")
save_report()

In [ ]:
# Phase A: clone (or update) the public repository and install it with fairchem.
if (REPO / ".git").exists():
    ok = sh("A_clone", ["git", "-C", REPO, "pull", "--ff-only"], cwd="/content")
else:
    ok = sh("A_clone", ["git", "clone", "--depth", "1", REPO_URL, REPO], cwd="/content")
if ok:
    sh("A_install", [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[dev]", "fairchem-core==2.24.0"])
try:
    REPORT["commit"] = subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip()
except Exception as e:
    REPORT["commit"] = f"inconnu ({type(e).__name__})"
save_report()
print("Commit testé :", REPORT["commit"])

In [ ]:
# Seed cache (259 UMA relaxations, MP references), then Google Drive backup: restore what a previous session finished.
t0 = time.time()
try:
    if not SEED.exists():
        from google.colab import files
        print("Choisir autolab_seed_step9.zip (D:\\autolab\\.cache\\autolab_seed_step9.zip sur le PC) :")
        for name, data in files.upload().items():
            SEED.write_bytes(data)
    with zipfile.ZipFile(SEED) as z:
        z.extractall(CACHE)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BACKUP.mkdir(parents=True, exist_ok=True)
        USE_BACKUP = True
        for sub in ("relax", "md"):
            if (BACKUP / sub).exists():
                shutil.copytree(BACKUP / sub, CACHE / sub, dirs_exist_ok=True)
    except Exception as e:
        print(f"Google Drive non disponible ({type(e).__name__}) : pas de sauvegarde, une déconnexion ferait tout perdre.")
    n_relax = len(list((CACHE / "relax").glob("*.json")))
    n_md = len(list((CACHE / "md").glob("*.json"))) if (CACHE / "md").exists() else 0
    record("B_seed", "ok", time.time() - t0, relaxations_in_cache=n_relax, md_in_cache=n_md, drive_backup=USE_BACKUP)
    print(f"Relaxations en cache : {n_relax} (au moins 259) ; dynamiques en cache : {n_md} ; sauvegarde Drive : {USE_BACKUP}")
except Exception as e:
    record("B_seed", f"erreur : {type(e).__name__}: {e}", time.time() - t0)

In [ ]:
# 9B: calibrations (UMA vs MP), then candidate interfaces with the cathode if both calibrations pass.
backup_args = ["--backup", BACKUP] if USE_BACKUP else []
sh("C_interfaces", [sys.executable, "scripts/run_interfaces.py", STEP_DIR / "candidates.json",
                    "--ids", *INTERFACE_IDS, "--cathode", CATHODE, "--max-relaxations", MAX_RELAXATIONS,
                    "--out", RUN_DIR, *backup_args])

In [ ]:
# 9C: molecular dynamics, parents first, each finished run cached (and saved to Drive).
sh("D_md", [sys.executable, "scripts/run_md_screen.py", STEP_DIR / "candidates.json", "--ids", *MD_IDS,
            "--parents", *MD_PARENTS, "--temperature", MD_TEMPERATURE_K, "--ps", MD_PS, "--out", RUN_DIR,
            *(["--backup", BACKUP] if USE_BACKUP else [])])

In [ ]:
# Final report (copy everything below) + zip of the results to download.
lines = ["# Rapport étape 9B + 9C — AutoLab sur Colab",
         f"GPU : {REPORT.get('gpu')} | Python {REPORT.get('python')} | commit {REPORT.get('commit')}",
         f"Secrets : {REPORT.get('secrets')} | paramètres : {REPORT.get('parameters')}", ""]
try:
    sh("E_report", [sys.executable, "scripts/step9_report.py", "--step-dir", STEP_DIR, "--run-dir", RUN_DIR,
                    "--out", RUN_DIR / "step9_report.md"])
    report_md = RUN_DIR / "step9_report.md"
    if report_md.exists():
        lines.append(report_md.read_text(encoding="utf-8"))
    for step in ("C_interfaces", "D_md"):
        log = WORK / f"{step}.log"
        if log.exists() and REPORT["steps"].get(step, {}).get("status") != "ok":
            lines += [f"**{step} n'a pas fini normalement.** Fin du journal :", "```",
                      *log.read_text(encoding="utf-8").splitlines()[-30:], "```"]
    lines += ["", "## Durées du notebook"]
    for step, info in REPORT["steps"].items():
        extra = {k: v for k, v in info.items() if k not in ("status", "seconds")}
        lines.append(f"- {step} : {info['status']}, {info['seconds'] / 60:.1f} min {extra or ''}")
    lines.append(f"- durée totale : {(time.time() - T_START) / 60:.1f} min")
except Exception as e:
    lines.append(f"ERREUR pendant le rapport : {type(e).__name__}: {e}")

text = "\n".join(lines)
(WORK / "report.md").write_text(text, encoding="utf-8")
save_report()
print(text)

# Light zip to download (relaxations, MP references, results, logs). The MD trajectories stay in the
# Drive backup (and in autolab_step9_md.zip on Colab) because of their size.
try:
    stage = Path("/content/autolab_step9_results")
    shutil.rmtree(stage, ignore_errors=True)
    shutil.copytree(CACHE / "relax", stage / "cache" / "relax")
    if (CACHE / "mp").exists():
        shutil.copytree(CACHE / "mp", stage / "cache" / "mp", ignore=shutil.ignore_patterns("entries"))
    shutil.copytree(WORK, stage / "run")
    archive = shutil.make_archive("/content/autolab_step9_results", "zip", stage)
    print(f"\nZip prêt : {archive} ({os.path.getsize(archive) / 1e6:.1f} Mo)")
    if (CACHE / "md").exists():
        md_zip = shutil.make_archive("/content/autolab_step9_md", "zip", CACHE / "md")
        print(f"Trajectoires : {md_zip} ({os.path.getsize(md_zip) / 1e6:.1f} Mo), aussi dans la sauvegarde Drive")
    from google.colab import files
    files.download(archive)
except Exception as e:
    print(f"Zip/téléchargement : {type(e).__name__}: {e} — le fichier reste dans le panneau Fichiers de Colab.")